# Magnificent Seven stock tracker

Run all cells to see the latest available daily prices for NVIDIA, Apple, Microsoft, Amazon, Alphabet, Meta, and Tesla. Change the chart settings below and rerun the chart cell to explore a different stock or period.

Requires Python with `requests`, `pandas`, and `altair`, plus a Jupyter-compatible notebook environment. The existing project environment already includes these data libraries. Data comes from Yahoo Finance's unofficial chart endpoint and may be delayed, unavailable, or rate-limited. Today's daily price can change while the market is open; closes are provider-reported and not dividend-adjusted. This notebook is for informational purposes, not investment advice.

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime, timezone

import altair as alt
import pandas as pd
import requests
from IPython.display import display

COMPANIES = {
    'NVDA': 'NVIDIA', 'AAPL': 'Apple', 'MSFT': 'Microsoft',
    'AMZN': 'Amazon', 'GOOGL': 'Alphabet', 'META': 'Meta', 'TSLA': 'Tesla',
}
PERIODS = {'1 month': 1, '3 months': 3, '6 months': 6, '1 year': 12}

def fetch_history(symbol):
    response = requests.get(
        f'https://query1.finance.yahoo.com/v8/finance/chart/{symbol}',
        params={'range': '1y', 'interval': '1d'},
        headers={'User-Agent': 'Mozilla/5.0'},
        timeout=10,
    )
    response.raise_for_status()
    chart = response.json()['chart']
    if chart.get('error'):
        raise ValueError(f"Market data error: {chart['error']}")
    result = chart['result'][0]
    timestamps = result['timestamp']
    closes = result['indicators']['quote'][0]['close']
    if len(timestamps) != len(closes):
        raise ValueError('Market data contains mismatched dates and prices')
    frame = pd.DataFrame({
        'Date': pd.to_datetime(timestamps, unit='s', utc=True)
            .tz_convert(result['meta']['exchangeTimezoneName']).date,
        'Close': pd.to_numeric(closes, errors='coerce'),
    }).dropna(subset=['Close'])
    frame = frame[frame['Close'] > 0].sort_values('Date')
    if frame.empty:
        raise ValueError('No usable daily prices returned')
    return frame

## Fetch and display prices

Rerun this cell to refresh the data. Change is relative to the previous available trading day's close.

In [ ]:
histories = {}
errors = {}
with ThreadPoolExecutor(max_workers=len(COMPANIES)) as executor:
    futures = {symbol: executor.submit(fetch_history, symbol) for symbol in COMPANIES}
    for symbol, future in futures.items():
        try:
            histories[symbol] = future.result()
        except (requests.RequestException, ValueError, KeyError, IndexError, TypeError) as exc:
            errors[symbol] = str(exc)

print(f"Data retrieved {datetime.now(timezone.utc):%Y-%m-%d %H:%M UTC}")
for symbol, error in errors.items():
    print(f'{symbol}: unavailable ({error})')
if not histories:
    raise RuntimeError('No market data is available. Rerun this cell to retry.')

rows = []
for symbol, name in COMPANIES.items():
    if symbol not in histories:
        rows.append({'Symbol': symbol, 'Company': name, 'Date': None, 'Price (USD)': None, 'Change (%)': None})
        continue
    prices = histories[symbol]
    latest = float(prices['Close'].iloc[-1])
    change = (latest / float(prices['Close'].iloc[-2]) - 1) * 100 if len(prices) >= 2 else None
    rows.append({'Symbol': symbol, 'Company': name, 'Date': prices['Date'].iloc[-1], 'Price (USD)': latest, 'Change (%)': change})
summary = pd.DataFrame(rows)
display(summary.style.format({'Price (USD)': '${:,.2f}', 'Change (%)': '{:+.2f}%'}, na_rep='Unavailable'))

## Price history

Edit `symbol` and `period` below, then rerun the cell. Available symbols are `NVDA`, `AAPL`, `MSFT`, `AMZN`, `GOOGL`, `META`, and `TSLA`; periods are `1 month`, `3 months`, `6 months`, and `1 year`.

In [ ]:
symbol = 'NVDA'
period = '3 months'

if symbol not in COMPANIES:
    raise ValueError(f'Unknown symbol {symbol!r}. Choose from {list(COMPANIES)}.')
if period not in PERIODS:
    raise ValueError(f'Unknown period {period!r}. Choose from {list(PERIODS)}.')
if symbol not in histories:
    raise RuntimeError(f'{symbol} is unavailable. Rerun the data cell to retry.')

history = histories[symbol].copy()
history['Date'] = pd.to_datetime(history['Date'])
cutoff = history['Date'].iloc[-1] - pd.DateOffset(months=PERIODS[period])
history = history[history['Date'] >= cutoff]
display(
    alt.Chart(history, title=f'{COMPANIES[symbol]} ({symbol}) daily prices')
    .mark_line()
    .encode(x=alt.X('Date:T', title='Date'), y=alt.Y('Close:Q', title='Price (USD)', scale=alt.Scale(zero=False)),
            tooltip=['Date:T', alt.Tooltip('Close:Q', format='$.2f')])
    .properties(width=700, height=350)
)